# How aggregation works — Overview

---

## 1  The TSA taxonomy: two axes, four cells

Time-series aggregation (TSA) methods can be classified along two independent axes
(Hoffmann et al. 2020, [Table 3](https://www.mdpi.com/1996-1073/13/3/641)):

* **How** periods or timesteps are grouped — by **time position** (consecutive
  blocks, calendar position) or by **feature similarity** (clustering by value).
* **What the result looks like** — **resolution variation** (fewer or coarser
  timesteps, same calendar extent) or **typical periods** (a small set of
  representative day-shapes with occurrence weights).

These two axes are independent, so methods from both columns can be
combined in a single pipeline (e.g., cluster periods *and* segment within each period).

|                   | **Resolution variation** | **Typical periods**          |
|-------------------|--------------------------|------------------------------|
| **Time-based**    | Downsampling             | Time slices / averaging      |
| **Feature-based** | Segmentation             | Clustering                   |

---

## 2  What tsam implements — and where it sits in the table

tsam's primary focus is the **feature-based / typical periods** cell: given a
time series split into equal-length periods (e.g. days), it finds $k$ representative
periods using clustering.

| Taxonomy cell | Method | tsam name | Notebook |
|---|---|---|---|
| Feature-based → Typical periods | K-means | `kmeans` | [01](01_partitional_clustering.ipynb) |
| Feature-based → Typical periods | K-medoids | `kmedoids` | [01](01_partitional_clustering.ipynb) |
| Feature-based → Typical periods | K-maxoids | `kmaxoids` | [01](01_partitional_clustering.ipynb) |
| Feature-based → Typical periods | Hierarchical Ward | `hierarchical` | [02](02_agglomerative_clustering.ipynb) |
| Feature-based → Typical periods | Contiguous Ward\* | `contiguous` | [02](02_agglomerative_clustering.ipynb) |
| Time-based → Typical periods | Block averaging | `averaging` | [03](03_averaging.ipynb) |
| Feature-based → Resolution variation | Segmentation | `SegmentConfig` | [04](04_segmentation.ipynb) |
| Cross-cutting | Representation & rescaling | `ClusterConfig(representation=...)` | [05](05_representation_rescaling.ipynb) |
| Cross-cutting | Extreme periods | `ExtremeConfig` | [06](06_extreme_periods.ipynb) |

**Nuances worth noting:**

* `contiguous`\* is Ward agglomerative clustering with a temporal-adjacency
  constraint — it is **feature-based** (driven by value similarity), not
  time-based. The adjacency constraint prevents non-neighbouring periods from
  merging, but the merge cost is still the Ward variance criterion.
* `averaging` produces **consecutive equal-size blocks** only. Full calendar
  time-slices (e.g. "all winter weekdays") are not built in; build the
  assignment vector externally if you need them.
* **Downsampling** (coarser timesteps, same calendar) is intentionally
  delegated to pandas: `df.resample(rule).mean()`. tsam does not duplicate this.

---

## 3  Further methods not in tsam

Several methods surveyed in [Hoffmann et al. (2020)](https://www.mdpi.com/1996-1073/13/3/641)
are not implemented in tsam. Downsampling is a one-liner in pandas and needs no
dedicated support. Full calendar time-slices (grouping by season and weekday type)
require an external assignment vector. Alternative centroid variants such as
k-medians and k-centers, shape-based and time-shift-tolerant methods (DTW,
k-shape), and dimensionality-reduction pre-processing (PCA, autoencoders) all
fall outside tsam's scope and are surveyed with references in Hoffmann et al.
(2020). Random period sampling and multiple-time-grid schemes (different
resolutions per season) are likewise not supported within a single aggregation call.

---

## The tiny synthetic dataset

Six days × 4 timesteps/day (6-hourly), two attributes: **solar** irradiance proxy and
**load** proxy. Day shapes are deliberately distinct:

* Days 0–1: bright sunny days (high solar peak)
* Days 2–3: overcast / moderate days
* Days 4–5: cloudy + one extreme-load day

This is small enough to verify every number by hand.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"

# --------------------------------------------------------------------------
# Tiny 6-day × 4-timestep synthetic dataset
# --------------------------------------------------------------------------
idx = pd.date_range("2020-01-01", periods=24, freq="6h")

solar_values = [
    0, 8, 6, 0,  # day 0 — sunny
    0, 7, 7, 0,  # day 1 — sunny
    0, 3, 2, 0,  # day 2 — overcast
    0, 2, 3, 0,  # day 3 — overcast
    0, 1, 1, 0,  # day 4 — cloudy
    0, 1, 0, 0,  # day 5 — cloudy + extreme load
]

load_values = [
    3, 3, 4, 5,   # day 0
    3, 3, 4, 4,   # day 1
    4, 5, 5, 6,   # day 2
    4, 4, 6, 5,   # day 3
    6, 6, 7, 7,   # day 4
    6, 7, 8, 10,  # day 5 — extreme demand
]

tiny = pd.DataFrame({"solar": solar_values, "load": load_values}, index=idx)
print("Shape:", tiny.shape, "  (6 days x 4 timesteps, 2 attributes)")
tiny

Shape: (24, 2)   (6 days x 4 timesteps, 2 attributes)


,solar,load
2020-01-01 00:00:00,0,3
2020-01-01 06:00:00,8,3
2020-01-01 12:00:00,6,4
2020-01-01 18:00:00,0,5
2020-01-02 00:00:00,0,3
2020-01-02 06:00:00,7,3
2020-01-02 12:00:00,7,4
2020-01-02 18:00:00,0,4
2020-01-03 00:00:00,0,4
2020-01-03 06:00:00,3,5


In [2]:
fig = px.line(
    tiny.reset_index().rename(columns={"index": "time"}),
    x="time",
    y=["solar", "load"],
    facet_col="variable",
    title="Tiny synthetic dataset (6 days x 4 timesteps)",
    labels={"value": "value", "time": "timestamp"},
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

---

## Shared preprocessing: normalization and unstacking to period row-vectors

Before any clustering can happen tsam runs two preprocessing steps (Hoffmann §3.2.2.1):

### Attribute-wise min-max normalization

Each attribute is scaled to $[0, 1]$ so no column dominates the distance:

$$
x_{a,s} = \frac{x'_{a,s} - \min x'_a}{\max x'_a - \min x'_a}
$$

where $x'_{a,s}$ is the raw value of attribute $a$ at time step $s$, and
$\min x'_a$, $\max x'_a$ are the column min/max over all time steps.

In [3]:
col_min = tiny.min()
col_max = tiny.max()

print("Column minimums:")
print(col_min.to_string())
print("\nColumn maximums:")
print(col_max.to_string())

normalized = (tiny - col_min) / (col_max - col_min)
print("\nNormalized dataset (all values in [0, 1]):")
normalized.round(4)

Column minimums:
solar    0
load     3

Column maximums:
solar     8
load     10

Normalized dataset (all values in [0, 1]):


,solar,load
2020-01-01 00:00:00,0.000,0.0000
2020-01-01 06:00:00,1.000,0.0000
2020-01-01 12:00:00,0.750,0.1429
2020-01-01 18:00:00,0.000,0.2857
2020-01-02 00:00:00,0.000,0.0000
2020-01-02 06:00:00,0.875,0.0000
2020-01-02 12:00:00,0.875,0.1429
2020-01-02 18:00:00,0.000,0.1429
2020-01-03 00:00:00,0.000,0.1429
2020-01-03 06:00:00,0.375,0.2857


### Unstacking to period row-vectors (the D matrix)

The flat normalized series is **reshaped** so each period (day) becomes one row
vector whose dimensions are:

$$
\text{dim} = N_t \times N_a \quad (\text{timesteps per period} \times \text{attributes})
$$

With 4 timesteps/day and 2 attributes, each period is an 8-dimensional point.
Clustering groups these six points in that 8-dimensional space.

In [4]:
import tsam

# tsam's unstack_to_periods does the same reshape
unstacked = tsam.unstack_to_periods(tiny, period_duration="1D")
print("Period matrix shape:", unstacked.shape, "(rows=days, cols=(attribute, timestep) pairs)")
unstacked.round(4)

Period matrix shape: (6, 8) (rows=days, cols=(attribute, timestep) pairs)


solar          load          
TimeStep      0  1  2  3    0  1  2   3
PeriodNum                              
0             0  8  6  0    3  3  4   5
1             0  7  7  0    3  3  4   4
2             0  3  2  0    4  5  5   6
3             0  2  3  0    4  4  6   5
4             0  1  1  0    6  6  7   7
5             0  1  0  0    6  7  8  10

In [5]:
N_TIMESTEPS = 4
N_PERIODS = 6
N_ATTRS = 2

norm_vals = normalized.values  # (24, 2)
D_arr = norm_vals.reshape(N_PERIODS, N_TIMESTEPS * N_ATTRS)

col_names = [
    f"{attr}_t{t}"
    for attr in ["solar", "load"]
    for t in range(N_TIMESTEPS)
]
D_df = pd.DataFrame(
    D_arr,
    columns=col_names,
    index=[f"day_{i}" for i in range(N_PERIODS)],
)
print("Period matrix D (shape:", D_arr.shape, "— each row is one 8-dim period):")
D_df.round(4)

Period matrix D (shape: (6, 8) — each row is one 8-dim period):


,solar_t0,solar_t1,solar_t2,solar_t3,load_t0,load_t1,load_t2,load_t3
day_0,0.0,0.0000,1.000,0.0000,0.750,0.1429,0.0,0.2857
day_1,0.0,0.0000,0.875,0.0000,0.875,0.1429,0.0,0.1429
day_2,0.0,0.1429,0.375,0.2857,0.250,0.2857,0.0,0.4286
day_3,0.0,0.1429,0.250,0.1429,0.375,0.4286,0.0,0.2857
day_4,0.0,0.4286,0.125,0.4286,0.125,0.5714,0.0,0.5714
day_5,0.0,0.4286,0.125,0.5714,0.000,0.7143,0.0,1.0000


---

## Notebook map

Each child notebook goes deep on one part of the pipeline:

* [Partitional clustering](01_partitional_clustering.ipynb) — k-means, k-medoids, k-maxoids
* [Agglomerative clustering](02_agglomerative_clustering.ipynb) — hierarchical Ward, contiguous Ward
* [Averaging](03_averaging.ipynb) — positional block-grouping
* [Segmentation](04_segmentation.ipynb) — reducing timesteps within periods
* [Representation & rescaling](05_representation_rescaling.ipynb) — mean/medoid/maxoid/distribution, rescaling formula
* [Extreme periods](06_extreme_periods.ipynb) — append/replace/new_cluster strategies

### Further reading

* [Mathematical Background](../../background/math.md) — full notation and formulas
* [Pipeline Guide](../../background/architecture/pipeline_guide.md) — the four pipeline phases
* [Clustering methods](../clustering_methods.ipynb) — accuracy and speed benchmarks